In [ ]:
%pip install torch
%pip install transformers
%pip install datasets
%pip install peft
%pip install bitsandbytes
%pip install accelerate
%pip install evaluate

In [ ]:
#imports
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from datasets import load_dataset, concatenate_datasets

In [ ]:
#Global Variables
LORA = True
QUANTIZE = False
CLUSTER = False
CUDA = True
CPU = False
DEBUG = False
MODEL = "Qwen/Qwen3-4B"
# MODEL = "Qwen/Qwen3-8B"

In [ ]:
device = torch.device("cuda" if CUDA and torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
tasks = ["abstract_algebra", "anatomy", "astronomy", "business_ethics", "clinical_knowledge", "college_biology", "college_chemistry", 
         "college_computer_science", "college_mathematics", "college_medicine", "college_physics", "computer_security", "conceptual_physics", 
         "econometrics", "electrical_engineering", "elementary_mathematics", "formal_logic", "global_facts", "high_school_biology", 
         "high_school_chemistry", "high_school_computer_science", "high_school_european_history", "high_school_geography", 
         "high_school_government_and_politics", "high_school_macroeconomics", "high_school_mathematics", "high_school_microeconomics", 
         "high_school_physics", "high_school_psychology", "high_school_statistics", "high_school_us_history", "high_school_world_history", 
         "human_aging", "human_sexuality", "international_law", "jurisprudence", "logical_fallacies", "machine_learning", "management", 
         "marketing", "medical_genetics", "miscellaneous", "moral_disputes", "moral_scenarios", "nutrition", "philosophy", "prehistory", 
         "professional_accounting", "professional_law", "professional_medicine", "professional_psychology", "public_relations", "security_studies", 
         "sociology", "us_foreign_policy", "virology", "world_religions"]


datasets = {}
for task in tasks:
    datasets[task] = load_dataset("cais/mmlu", task, split="test")

In [ ]:
for question in datasets["world_religions"]:
    if question["answer"] == 0:
        print("found 4")


In [ ]:
#dataset generation
def format(training_example):
    prompt = (
        "Given the following subject and question, choose the correct answer from the provided options.\n\n"
        f"Subject: {training_example['subject']}\n"
        f"Question: {training_example['question']}\n"
        f"Answer Choices: {training_example['choices']}\n"
        f"Pick a number 0-3 corresponding to the correct answer choice.\n\n"
    )
    answer = str(training_example["answer"])

    if isinstance(answer, list):
        answer = answer[0]
    elif answer == [""] or "":
        raise ValueError("Empty answer found in the dataset.")

    return {"prompt": prompt, "response": answer}

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(training_example):
    prompt_tokens = tokenizer(training_example["prompt"], padding="max_length", truncation=True)['input_ids']
    if DEBUG:
        print(type(prompt_tokens))
        print(prompt_tokens)
    answer_tokens = tokenizer(training_example["response"], padding="max_length", truncation=True)['input_ids']
    if DEBUG:
        print(type(answer_tokens))
        print(answer_tokens)
    input_ids = prompt_tokens + answer_tokens + [tokenizer.eos_token_id]
    if DEBUG:
        print(type(input_ids))
        print(input_ids)
    labels = [-100] * len(prompt_tokens) + answer_tokens + [tokenizer.eos_token_id]
    if DEBUG:
        print(type(labels))
        print(labels)
    attention_mask = [1] * len(input_ids)
    if DEBUG:
        print(type(attention_mask))
        print(attention_mask)
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

In [ ]:
train_datasets = {}
for task_name, ds in datasets.items():
    train_datasets[task_name] = ds.map(format, remove_columns=ds.column_names)
all_train_dataset = concatenate_datasets(list(train_datasets.values()))
tokenized_dataset = all_train_dataset.map(tokenize, remove_columns=all_train_dataset.column_names)

In [ ]:
train, test = tokenized_dataset.train_test_split(test_size=0.1).values()

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import bitsandbytes as bnb

model = None
print("Loading model...")

model = None
bnb_config = None

if QUANTIZE:
    print("  Applying 4-bit quantization config...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config if QUANTIZE else None,
    device_map="auto" if CUDA else None,
    trust_remote_code=True
)

if LORA:
    print("  Adding LoRA adapters...")
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(base_model, lora_config)
else:
    model = base_model

model.config.use_cache = False

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

print("Model ready ✅\n")
print(model)

for name, module in model.named_modules():
    print(name)


In [ ]:
from evaluate import load

metric = load("rouge")

def compute_metrics(predictions, references):
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_refs = tokenizer.batch_decode(references, skip_special_tokens=True)
    result = metric.compute(predictions=decoded_preds, references=decoded_refs, use_stemmer=True)
    result = {key: value.mid.fmeasure * 100 for key, value in result.items()}
    return result

In [ ]:
from transformers import TrainingArguments

mode = []
if QUANTIZE and LORA:
    mode.append("quantized_lora")
if QUANTIZE:
    mode.append("quantized")
if LORA:
    mode.append("lora")
if not mode:
    mode.append("fp")  # full precision, no quantization, no LoRA

output_dir = f"./Qwen3-8B-LongBench-finetuned-{'_'.join(mode)}"

training_args = TrainingArguments(
    output_dir=output_dir,
    do_eval=True,
    eval_strategy="steps",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    gradient_checkpointing=True,
    save_steps=600,
    eval_steps=300,
    logging_steps=300,
    learning_rate=5e-5,
    warmup_steps=500,
    save_total_limit=2,
    push_to_hub=False,
    use_cpu=CPU
)

from transformers import Trainer

trainer = Trainer(
    model=model,
    data_collator=lambda data: {
        'input_ids': torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(f['input_ids']) for f in data], batch_first=True, padding_value=tokenizer.pad_token_id
        ),
        'attention_mask': torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(f['attention_mask']) for f in data], batch_first=True, padding_value=0
        ),
        'labels': torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(f['labels']) for f in data], batch_first=True, padding_value=-100
        ),
    },
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train,
    eval_dataset=test,
    tokenizer=tokenizer
)



In [ ]:
trainer.train()